In [6]:
import pandas as pd

model = pd.read_csv("/bay_area_modeling_table.csv", low_memory=False)
dash = pd.read_csv("/dashboard_data.csv", low_memory=False)
eth = pd.read_csv("/uc_admissions_summary_by_ethnicity.csv")
disc = pd.read_csv("/uc_freshman_admission_by_discipline.csv")
transfer = pd.read_csv("/uc_transfer_admission_by_major.csv")

print("model:", model.shape)
print("dashboard:", dash.shape)
print("ethnicity:", eth.shape)
print("discipline:", disc.shape)
print("transfer:", transfer.shape)

model: (34311, 65)
dashboard: (34311, 72)
ethnicity: (4239, 6)
discipline: (101, 12)
transfer: (49, 13)


In [7]:
q1 = eth[
    (eth["fall_term"] == 2025) &
    (eth["entrant_level"] == "freshman") &
    (eth["count_type"] == "App")
]

# Unique people who applied somewhere in UC
unique_applicants = q1[q1["campus"] == "Systemwide"]["n"].sum()

# Total campus applications
campus_applications = q1[q1["campus"] != "Systemwide"]["n"].sum()

average_campuses = campus_applications / unique_applicants

print("Unique applicants:", unique_applicants)
print("Campus applications:", campus_applications)
print("Average campuses:", round(average_campuses, 2))

Unique applicants: 205389
Campus applications: 932623
Average campuses: 4.54


In [8]:
overall = disc[
    disc["broad_discipline"] == "All disciplines"
][["campus", "admit_rate"]].rename(
    columns={"admit_rate": "overall_rate"}
)

cs = disc[
    disc["broad_discipline"] == "Computer Science"
][["campus", "admit_rate"]].rename(
    columns={"admit_rate": "cs_rate"}
)

comparison = overall.merge(cs, on="campus")

comparison["admit_rate_cost"] = (
    comparison["overall_rate"] - comparison["cs_rate"]
)

comparison.sort_values(
    "admit_rate_cost",
    ascending=False
)

,campus,overall_rate,cs_rate,admit_rate_cost
1,Davis,0.44,0.19,0.25
5,San Diego,0.28,0.20,0.08
4,Riverside,0.87,0.81,0.06
0,Berkeley,0.11,0.06,0.05
6,Santa Barbara,0.38,0.34,0.04
3,Los Angeles,0.09,0.07,0.02
2,Irvine,0.29,0.28,0.01
7,Santa Cruz,0.72,0.79,-0.07


In [9]:
answers = {}

answers["Q1_avg_campuses"] = 4.54
answers["Q3_cs_penalty_campus"] = "Davis"
answers["Q3_cs_penalty"] = 0.25

answers

{'Q1_avg_campuses': 4.54,
 'Q3_cs_penalty_campus': 'Davis',
 'Q3_cs_penalty': 0.25}

In [10]:
comparison["overall_pct"] = comparison["overall_rate"] * 100
comparison["cs_pct"] = comparison["cs_rate"] * 100
comparison["penalty_pp"] = comparison["admit_rate_cost"] * 100

comparison.sort_values("penalty_pp", ascending=False)[
    ["campus", "overall_pct", "cs_pct", "penalty_pp"]
]

,campus,overall_pct,cs_pct,penalty_pp
1,Davis,44.0,19.0,25.0
5,San Diego,28.0,20.0,8.0
4,Riverside,87.0,81.0,6.0
0,Berkeley,11.0,6.0,5.0
6,Santa Barbara,38.0,34.0,4.0
3,Los Angeles,9.0,7.0,2.0
2,Irvine,29.0,28.0,1.0
7,Santa Cruz,72.0,79.0,-7.0


In [11]:
berkeley_cs = disc[
    (disc["campus"] == "Berkeley") &
    (disc["broad_discipline"] == "Computer Science")
]

berkeley_cs[
    ["admit_gpa_p25", "admit_gpa_p75"]
]

,admit_gpa_p25,admit_gpa_p75
4,4.2,4.29


In [12]:
e2025 = eth[
    (eth["fall_term"] == 2025) &
    (eth["entrant_level"] == "freshman") &
    (eth["ethnicity"].isin(["White", "Hispanic/Latino(a)"]))
]

rates = e2025.pivot_table(
    index=["campus", "ethnicity"],
    columns="count_type",
    values="n",
    aggfunc="sum"
).reset_index()

rates["admit_rate"] = rates["Adm"] / rates["App"]

rates[["campus", "ethnicity", "App", "Adm", "admit_rate"]]

count_type,campus,ethnicity,App,Adm,admit_rate
0,Berkeley,Hispanic/Latino(a),25084,2962,0.118083
1,Berkeley,White,21690,2607,0.120194
2,Davis,Hispanic/Latino(a),22248,7980,0.358684
3,Davis,White,16660,7503,0.450360
4,Irvine,Hispanic/Latino(a),32887,6127,0.186305
5,Irvine,White,16519,4540,0.274835
6,Los Angeles,Hispanic/Latino(a),34075,2566,0.075304
7,Los Angeles,White,25094,2509,0.099984
8,Merced,Hispanic/Latino(a),19235,18289,0.950819
9,Merced,White,4925,4773,0.969137


In [13]:
compare_eth = rates.pivot(
    index="campus",
    columns="ethnicity",
    values="admit_rate"
)

campuses_only = compare_eth.drop(index="Systemwide")

campuses_only["White_higher"] = (
    campuses_only["White"] >
    campuses_only["Hispanic/Latino(a)"]
)

campuses_only

ethnicity,Hispanic/Latino(a),White,White_higher
campus,,,
Berkeley,0.118083,0.120194,True
Davis,0.358684,0.450360,True
Irvine,0.186305,0.274835,True
Los Angeles,0.075304,0.099984,True
Merced,0.950819,0.969137,True
Riverside,0.832656,0.902254,True
San Diego,0.259083,0.279440,True
Santa Barbara,0.308506,0.381941,True
Santa Cruz,0.619164,0.791047,True


In [14]:
campuses_only["White_higher"].sum()

np.int64(9)

In [15]:
q7 = model[
    (model["fall_term"] == 2023) &
    (model["campus"] == "Universitywide")
]

ccc = q7["enrolled_ccc"].sum()
graduates = q7["hs_completers"].sum()

share = ccc / graduates

print("CCC enrollees:", ccc)
print("HS completers:", graduates)
print("Share:", share)
print("Percent:", round(share * 100, 2))

CCC enrollees: 21644.0
HS completers: 64345.0
Share: 0.3363742326521097
Percent: 33.64


In [16]:
mission = model[
    (model["fall_term"] == 2023) &
    (model["campus"] == "Universitywide") &
    (model["high_school"] == "MISSION SAN JOSE HIGH SCHOOL")
]

mission[
    ["high_school", "applicants", "ag_completers"]
]

,high_school,applicants,ag_completers
29666,MISSION SAN JOSE HIGH SCHOOL,420.0,424.0


In [17]:
share = (
    mission["applicants"].iloc[0]
    / mission["ag_completers"].iloc[0]
)

print(share)
print(round(share * 100, 2))

0.9905660377358491
99.06


In [18]:
choices = [
    "HERCULES HIGH SCHOOL",
    "MISSION SENIOR HIGH SCHOOL",
    "MONTEREY TRAIL HIGH SCHOOL",
    "PHILLIP & SALA BURTON ACAD HS",
    "RANCHO SAN JUAN HIGH SCHOOL"
]

q10 = dash[
    (dash["campus"] == "Berkeley") &
    (dash["fall_term"].between(2022, 2025)) &
    (dash["high_school"].isin(choices))
]

q10[
    [
        "fall_term",
        "high_school",
        "admit_rate",
        "expected_admit_rate",
        "admit_rate_residual"
    ]
].sort_values(["high_school", "fall_term"])

,fall_term,high_school,admit_rate,expected_admit_rate,admit_rate_residual
25385,2022,HERCULES HIGH SCHOOL,0.176471,NaN,NaN
27615,2023,HERCULES HIGH SCHOOL,0.181818,0.185409,-0.003590
29854,2024,HERCULES HIGH SCHOOL,0.377778,0.184858,0.192919
32119,2025,HERCULES HIGH SCHOOL,0.222222,0.185164,0.037058
25438,2022,MISSION SENIOR HIGH SCHOOL,0.348315,NaN,NaN
27669,2023,MISSION SENIOR HIGH SCHOOL,0.433333,0.166917,0.266416
29910,2024,MISSION SENIOR HIGH SCHOOL,0.371795,0.168921,0.202874
32176,2025,MISSION SENIOR HIGH SCHOOL,0.447368,0.166765,0.280604
25464,2022,PHILLIP & SALA BURTON ACAD HS,0.175258,NaN,NaN
27695,2023,PHILLIP & SALA BURTON ACAD HS,0.081081,0.168817,-0.087736


In [19]:
q10_summary = (
    q10.groupby("high_school")["admit_rate_residual"]
       .agg(["mean", "count"])
       .sort_values("mean", ascending=False)
)

q10_summary

,mean,count
high_school,,
MISSION SENIOR HIGH SCHOOL,0.249964,3
HERCULES HIGH SCHOOL,0.075462,3
PHILLIP & SALA BURTON ACAD HS,-0.106386,3


In [20]:
q9 = model[
    (model["fall_term"] == 2025) &
    (model["campus"] == "Universitywide") &
    (model["applicants"] >= 1)
]

print("Rows:", len(q9))
print("Unique school names:", q9["high_school"].nunique())
print("Unique ATP school IDs:", q9["atp_code"].nunique())

Rows: 248
Unique school names: 244
Unique ATP school IDs: 248


In [21]:
q9[
    q9.duplicated("high_school", keep=False)
][["high_school", "city", "county", "atp_code", "applicants"]].sort_values("high_school")

,high_school,city,county,atp_code,applicants
34063,ABRAHAM LINCOLN HIGH SCHOOL,San Francisco,San Francisco,52910,269.0
34064,ABRAHAM LINCOLN HIGH SCHOOL,San Jose,Santa Clara,53075,86.0
34142,FREMONT HIGH SCHOOL,Oakland,Alameda,52205,41.0
34143,FREMONT HIGH SCHOOL,Sunnyvale,Santa Clara,53460,190.0
34161,INDEPENDENCE HIGH SCHOOL,San Francisco,San Francisco,52998,20.0
34162,INDEPENDENCE HIGH SCHOOL,San Jose,Santa Clara,53087,112.0
34169,JOHN F KENNEDY HIGH SCHOOL,Fremont,Alameda,50966,99.0
34170,JOHN F KENNEDY HIGH SCHOOL,Richmond,Contra Costa,52630,24.0


In [22]:
q2 = model[
    (model["fall_term"] == 2025) &
    (model["campus"] == "Los Angeles")
]

total_applicants = q2["applicants"].sum()
total_admits = q2["admits"].sum()

admit_rate = total_admits / total_applicants

print("Applicants:", total_applicants)
print("Admits:", total_admits)
print("Admit rate:", admit_rate)
print("Percent:", round(admit_rate * 100, 2))

Applicants: 18516.0
Admits: 1514.0
Admit rate: 0.08176712032836465
Percent: 8.18


In [23]:
berkeley = dash[
    (dash["campus"] == "Berkeley") &
    (dash["fall_term"].between(2023, 2025)) &
    (dash["expected_admit_rate"].notna()) &
    (dash["applicants"].notna()) &
    (dash["admits"].notna())
].copy()

# Turn each school's expected rate into an expected number of admits
berkeley["expected_admits"] = (
    berkeley["expected_admit_rate"] *
    berkeley["applicants"]
)

summary = berkeley.groupby(
    "high_school",
    as_index=False
).agg(
    total_applicants=("applicants", "sum"),
    total_admits=("admits", "sum"),
    expected_admits=("expected_admits", "sum"),
    years=("fall_term", "nunique")
)

summary["actual_rate"] = (
    summary["total_admits"] /
    summary["total_applicants"]
)

summary["expected_rate"] = (
    summary["expected_admits"] /
    summary["total_applicants"]
)

summary["outperformance_pp"] = (
    summary["actual_rate"] -
    summary["expected_rate"]
) * 100

# Require at least 2 years of observations and 50 total applicants
stable = summary[
    (summary["years"] >= 2) &
    (summary["total_applicants"] >= 50)
].sort_values(
    "outperformance_pp",
    ascending=False
)

stable.head(15)

,high_school,total_applicants,total_admits,expected_admits,years,actual_rate,expected_rate,outperformance_pp
113,MISSION SENIOR HIGH SCHOOL,244.0,102.0,40.872533,3,0.418033,0.167510,25.052241
126,OAKLAND CHARTER HIGH SCHOOL,139.0,42.0,26.920796,3,0.302158,0.193675,10.848348
89,LEADERSHIP PUBLIC SCH RICHMOND,96.0,28.0,18.500609,3,0.291667,0.192715,9.895199
15,ANTIOCH HIGH SCHOOL,67.0,16.0,10.599702,2,0.238806,0.158205,8.060146
22,ASPIRE RICHMOND CA COLG PREP,83.0,23.0,16.934469,3,0.277108,0.204030,7.307869
47,DOZIER-LIBBEY MEDICAL HIGH SCH,75.0,19.0,13.839577,3,0.253333,0.184528,6.880564
69,HERCULES HIGH SCHOOL,154.0,39.0,28.514980,3,0.253247,0.185162,6.808455
52,EL CERRITO HIGH SCHOOL,359.0,77.0,56.422642,3,0.214485,0.157166,5.731855
167,SUMMIT PUBLIC SCHOOL K2,62.0,16.0,13.456880,2,0.258065,0.217046,4.101806
178,VINTAGE HIGH SCHOOL,130.0,25.0,19.759140,3,0.192308,0.151993,4.031431


In [24]:
# Count how many years each school beat expectations
consistency = berkeley.groupby("high_school").agg(
    total_applicants=("applicants", "sum"),
    total_admits=("admits", "sum"),
    expected_admits=("expected_admits", "sum"),
    years=("fall_term", "nunique"),
    positive_years=("admit_rate_residual", lambda x: (x > 0).sum())
).reset_index()

consistency["actual_rate"] = (
    consistency["total_admits"] /
    consistency["total_applicants"]
)

consistency["expected_rate"] = (
    consistency["expected_admits"] /
    consistency["total_applicants"]
)

consistency["outperformance_pp"] = (
    consistency["actual_rate"] -
    consistency["expected_rate"]
) * 100

consistent = consistency[
    (consistency["years"] == 3) &
    (consistency["positive_years"] == 3) &
    (consistency["total_applicants"] >= 50)
].sort_values("outperformance_pp", ascending=False)

consistent.head(10)

,high_school,total_applicants,total_admits,expected_admits,years,positive_years,actual_rate,expected_rate,outperformance_pp
113,MISSION SENIOR HIGH SCHOOL,244.0,102.0,40.872533,3,3,0.418033,0.167510,25.052241
126,OAKLAND CHARTER HIGH SCHOOL,139.0,42.0,26.920796,3,3,0.302158,0.193675,10.848348
89,LEADERSHIP PUBLIC SCH RICHMOND,96.0,28.0,18.500609,3,3,0.291667,0.192715,9.895199
22,ASPIRE RICHMOND CA COLG PREP,83.0,23.0,16.934469,3,3,0.277108,0.204030,7.307869
52,EL CERRITO HIGH SCHOOL,359.0,77.0,56.422642,3,3,0.214485,0.157166,5.731855
42,DE ANZA HIGH SCHOOL,166.0,34.0,28.117625,3,3,0.204819,0.169383,3.543599
124,NOVATO HIGH SCHOOL,173.0,32.0,27.497419,3,3,0.184971,0.158945,2.602648
91,LELAND HIGH SCHOOL,566.0,101.0,88.671519,3,3,0.178445,0.156663,2.178177
141,PITTSBURG HIGH SCHOOL,343.0,50.0,45.423273,3,3,0.145773,0.132429,1.334323
0,ABRAHAM LINCOLN HIGH SCHOOL,773.0,121.0,115.746810,3,3,0.156533,0.149737,0.679585
